# RAG vs Agents and Tool Use

RAG retrieves **static or semi-static documents** and synthesizes answers. **Agents** run a loop: plan → act (call tools) → observe → repeat until done. **Tools** connect the LLM to **live systems** — APIs, databases, calculators, code interpreters — capabilities RAG alone cannot provide.

This closing notebook contrasts RAG, chains, and agents; shows **hybrid architectures** (router + RAG + tools); covers when to add tool calling vs stay retrieval-only; and maps patterns to the **FastAPI Notes API** assistant. It completes the **06. Retrieval Augmented Generation** arc.

Prerequisites: **11. Conversational and Multi-Turn RAG**, **03. RAG Architecture Patterns**, **10. Production Patterns**, **04. Prompt Engineering** (structured output / tool calling). Next section in curriculum: **AI Agents & Agentic Workflows**.

In [ ]:
import json
from dataclasses import dataclass
from typing import Callable

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

print("Setup OK")

---

## 1. Three Abstractions: Chain, RAG, Agent

From notebook **03**, modern assistants combine these patterns:

| Pattern | Control flow | Data source | Example |
|---------|--------------|-------------|--------|
| **Chain** | Fixed steps you code | Prompt + optional API | "Summarize → translate" |
| **RAG** | Fixed retrieve → generate | Indexed documents | "How does JWT work?" |
| **Agent** | LLM chooses next action | Tools + optional RAG | "Create note + cite docs" |

```
Chain:   input ──► step A ──► step B ──► output     (deterministic code)
RAG:     query ──► retrieve ──► generate            (fixed pipeline **02**)
Agent:   goal  ──► [think ──► tool ──► observe]* ──► output   (LLM loop)
```

**Most production systems are hybrids** — a router picks RAG, a tool path, or bare chat per request.

---

## 2. Definitions Side by Side

| Dimension | **RAG** | **Tool use** | **Agent loop** |
|-----------|---------|--------------|----------------|
| **Primary data** | Indexed docs (c1–c5) | Live APIs / DB | Tools + optional docs |
| **Control flow** | Fixed pipeline | Single call per tool invoke | Multi-step LLM-driven |
| **Side effects** | Read-only | Can read/write | Can chain writes |
| **Latency** | ~1 embed + 1 LLM | +1 HTTP per tool | N × (LLM + tools) |
| **Predictability** | High | Medium | Lower |
| **Best for** | Docs Q&A, policies | One live lookup | Multi-step tasks |

### 2.1 RAG Is Not "Smarter Search"

RAG **generates** prose from retrieved evidence. Search returns links. RAG is for **synthesis** — "explain migration workflow in three steps" — not just finding `docs/migrations.md`.

---

## 3. Decision Framework

```
User request
     │
     ▼
Answer in static docs? ──yes──► RAG (**02–11**)
     │ no
     ▼
Need live / transactional data? ──yes──► Tool or API call
     │ no
     ▼
Multi-step with branching? ──yes──► Agent loop
     │ no
     ▼
Pure reasoning / chat ──► LLM only (no retrieve, no tools)
```

| User ask (Notes API) | Pattern |
|----------------------|--------|
| "How do I authenticate?" | **RAG** → c3 |
| "List my notes" | **Tool** → `GET /notes` |
| "Explain JWT docs then fetch note 42" | **Hybrid** → RAG + tool |
| "Create a note titled Deploy" | **Tool** (write) |
| "Summarize this paragraph" | **LLM only** (user pasted text) |

---

## 4. What RAG Does Well (This Corpus)

| Task | Chunks | Why RAG |
|------|--------|--------|
| Explain JWT header format | c3 | Document-grounded |
| Alembic migration workflow | c2, c5 | Multi-chunk synthesis |
| Pytest fixture conventions | c4 | Internal runbook |
| FastAPI route overview | c1 | API docs |

No **write access** required; answers are **read-only** over the knowledge base. Eval with **09**; deploy with **10**.

---

## 5. What Tools Add

Tools are **functions** the LLM can invoke with structured arguments (OpenAI **function calling** / **tools API**, **04. Prompt Engineering**).

| Tool | Example | Side effect? |
|------|---------|-------------|
| `search_docs` | Vector search over KB | Read |
| `get_note(note_id)` | `GET /notes/{id}` | Read |
| `create_note(title, body)` | `POST /notes` | **Write** |
| `run_sql(query)` | Count rows in DB | Read (if scoped) |
| `calculator(expr)` | Pricing math | None |

Tools need **auth**, **JSON schemas**, **timeouts**, and **idempotency** for writes — production concerns beyond RAG (**10**).

In [ ]:
# Illustrative tool schemas (OpenAI tools format)
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "Search the Notes API documentation knowledge base.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "Search query"}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_note",
            "description": "Fetch a single note by id from the live Notes API.",
            "parameters": {
                "type": "object",
                "properties": {"note_id": {"type": "integer"}},
                "required": ["note_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_note",
            "description": "Create a new note via POST /notes.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "body": {"type": "string"},
                },
                "required": ["title", "body"],
            },
        },
    },
]

print("Registered tools:", [t["function"]["name"] for t in TOOLS])

---

## 6. RAG as a Tool Inside an Agent

The cleanest hybrid: expose retrieval as **`search_docs(query)`** — the agent loop decides **when** to search.

```
Agent loop:
  LLM: I need doc info → call search_docs("JWT bearer")
  Tool returns: [c3 text]
  LLM: I need live data → call get_note(42)
  Tool returns: {title: "Deploy", ...}
  LLM: Final answer combining both
```

| Approach | Pros | Cons |
|----------|------|------|
| **Fixed RAG pipeline** | Simple, fast, testable | Always retrieves |
| **RAG as tool** | Retrieve only when needed | Extra LLM turns |
| **Agentic RAG (**03**)** | Corrective / graded retrieval | Complex |

Pure ReAct agents without `search_docs` are **weak on 200-page PDFs** — add retrieval as a tool.

---

## 7. Hybrid Architecture — Router + RAG + Tools

```
User question
     │
     ▼
 Intent router (rules or small LLM)
   ┌────────┼────────┐
   ▼        ▼        ▼
 RAG     Tool(s)   Chat only
 path    path      path
   └────────┼────────┘
            ▼
     Merge / format response
```

**Start with explicit routers** before fully autonomous agents — easier to test and safer for writes.

In [ ]:
def route_intent(question: str) -> str:
    """Rule-based intent router for teaching — replace with LLM classifier in prod."""
    q = question.lower()
    write_signals = ["create", "delete", "update", "post", "add note", "remove"]
    live_signals = ["list my notes", "fetch note", "get note", "note id", "live"]
    doc_signals = ["how", "what", "explain", "docs", "alembic", "jwt", "pytest", "migrate", "authenticate"]
    if any(s in q for s in write_signals):
        return "tool_write"
    if any(s in q for s in live_signals):
        return "tool_read"
    if any(s in q for s in doc_signals):
        return "rag"
    return "chat"


SAMPLES = [
    "How do JWT bearer tokens work?",
    "Create a new note titled Deploy",
    "List my notes from the API",
    "Explain Alembic migrations and fetch note 42",
    "Tell me a joke about databases",
]

for s in SAMPLES:
    intent = route_intent(s)
    print(f"{intent:12s} <- {s}")

In [ ]:
@dataclass
class HybridResult:
    intent: str
    rag_answer: str | None = None
    tool_result: dict | None = None
    final: str = ""


def mock_rag(q: str) -> str:
    if "jwt" in q.lower() or "auth" in q.lower():
        return "[RAG] Use JWT bearer tokens in Authorization header (c3)."
    if "alembic" in q.lower() or "migrat" in q.lower():
        return "[RAG] Run alembic upgrade head (c2)."
    return "[RAG] No relevant docs."


def mock_get_note(note_id: int) -> dict:
    return {"id": note_id, "title": "Deploy checklist", "body": "..."}


def handle_hybrid(question: str) -> HybridResult:
    intent = route_intent(question)
    result = HybridResult(intent=intent)
    parts = []
    if intent in ("rag", "tool_read", "tool_write") and any(w in question.lower() for w in ["explain", "how", "jwt", "alembic", "docs"]):
        result.rag_answer = mock_rag(question)
        parts.append(result.rag_answer)
    if intent in ("tool_read", "tool_write") or "fetch note" in question.lower() or "note 42" in question.lower():
        result.tool_result = mock_get_note(42)
        parts.append(f"[Tool] Note {result.tool_result['id']}: {result.tool_result['title']}")
    if intent == "chat":
        parts.append("[Chat] General response without retrieval.")
    result.final = " ".join(parts) if parts else "[No op]"
    return result


hybrid_q = "Explain JWT docs and fetch note 42"
hr = handle_hybrid(hybrid_q)
print(json.dumps({"intent": hr.intent, "final": hr.final}, indent=2))

---

## 8. The ReAct Loop (Conceptual)

**ReAct** = **Reason** + **Act**: the model interleaves thinking and tool calls.

```
Thought: User wants note 42 and JWT docs. Search docs first.
Action: search_docs("JWT authentication")
Observation: [c3] JWT bearer tokens...
Thought: Now fetch live note.
Action: get_note(42)
Observation: {title: "Deploy checklist", ...}
Thought: I can answer.
Action: finish(answer)
```

| Guardrail | Purpose |
|-----------|--------|
| `max_steps` (e.g. 5) | Prevent runaway loops |
| Tool allow-list | No arbitrary shell |
| Human approval on writes | Safety |
| Timeout per step | SLA (**10**) |

---

## 9. Reliability, Safety, and Cost

| Risk | RAG | Agent + tools |
|------|-----|---------------|
| Wrong fact | Hallucination / bad retrieve (**08**) | Same + wrong tool args |
| Data exfiltration | Corpus ACL (**04**) | Tool permissions |
| Destructive action | Rare (read-only) | `DELETE`, `POST` without guard |
| Runaway loop | Uncommon | Cap `max_steps` |
| Cost predictability | ~2 API calls | N LLM + M tool calls |

In [ ]:
# Illustrative latency: RAG vs single-tool vs 3-step agent
patterns = ["RAG (1x)", "RAG + 1 tool", "Agent (3 steps)"]
latency_ms = np.array([950, 1400, 3200])  # illustrative
api_calls = np.array([2, 3, 7])

fig, ax1 = plt.subplots(figsize=(8, 4))
x = np.arange(len(patterns))
ax1.bar(x - 0.2, latency_ms, width=0.4, label="Latency (ms)", color="#4c72b0")
ax2 = ax1.twinx()
ax2.bar(x + 0.2, api_calls, width=0.4, label="API calls", color="#c44e52", alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(patterns)
ax1.set_ylabel("Latency (illustrative)")
ax2.set_ylabel("API / tool calls")
ax1.set_title("Agents trade predictability for capability")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

---

## 10. Framework and Integration Landscape

| Style | Idea | When |
|-------|------|------|
| **Custom FastAPI** | You own router + RAG + tools (**10**) | Full control, learning |
| **OpenAI tools API** | Native function calling (**03** LLM API) | Standard tool schemas |
| **LangGraph / workflows** | Explicit state machine | Complex branching |
| **MCP (Model Context Protocol)** | Standardized tool servers | Interop across clients |

This curriculum built RAG **without** a framework first — you can add LangChain/LangGraph later as **orchestration**, not as a substitute for understanding retrieve → prompt → generate.

---

## 11. Migration Path for the Notes API Assistant

| Stage | Capability | Notebooks |
|-------|------------|----------|
| **v1** | Naive RAG on static markdown | **02**, **04** |
| **v2** | Eval gates + production API | **09**, **10** |
| **v3** | Multi-turn + query rewrite | **11** |
| **v4** | Read-only tools (`search_docs`, `get_note`) | This notebook |
| **v5** | Write tools + human approval | Agents section |

Ship **v1–v2** before agents — most internal doc bot value is **grounded read-only answers** (~80% RAG, ~20% tools in mature products).

---

## 12. When Not to Use RAG or Agents

Skip both when **deterministic lookup** suffices:

| Problem | Solution |
|---------|----------|
| Error code → help URL | Static lookup table |
| Primary key fetch | SQL `WHERE id = ?` |
| Exact SKU match | Search index |

Do not put an LLM in the hot path if regex and a database solve it in microseconds.

---

## 13. Section 06 — Curriculum Recap

```
01 Define RAG ──► 02 Naive pipeline ──► 03 Architecture patterns
       ──► 04 Ingest / KB ──► 05 Prompting ──► 06 Context assembly
       ──► 07 Advanced retrieval ──► 08 Failure modes & grounding
       ──► 09 End-to-end eval ──► 10 Production ──► 11 Multi-turn
       ──► 12 RAG vs Agents (this notebook)
```

**Foundation (section 05):** embeddings, chunking, vector stores, retrieval metrics.

**Upstream:** **03** LLM APIs, **04** prompt engineering.

**Downstream:** **AI Agents & Agentic Workflows**, LangChain/orchestration, production capstone.

---

## 14. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Agent for every query | Cost, latency, flakiness | Router → RAG default |
| No `search_docs` tool in agent | Poor long-doc coverage | RAG as tool |
| Write tools without approval | Data loss | Human-in-loop |
| Skip RAG eval before agents | Debug harder | **09** first |
| Framework before basics | Opaque failures | **02** pipeline first |
| Same auth for tools and user | Privilege escalation | Scoped credentials |

---

## 15. Summary

**RAG** grounds answers in **documents** — fixed retrieve → augment → generate. **Tools** connect to **live systems** with structured side effects. **Agents** loop until the task is done, choosing when to retrieve and when to act.

Use RAG for doc Q&A; add tools for live data and writes; use agents only when multi-step branching justifies the cost and risk. Hybrid **routers** with `search_docs` as a tool are the practical production pattern.

Demonstrations covered tool schemas, intent routing, hybrid mock handlers, and latency trade-offs.

You have completed **06. Retrieval Augmented Generation** — from definition (**01**) through production (**10**) and conversational RAG (**11**) to the agents boundary (**12**).

Back: **11. Conversational and Multi-Turn RAG**. Continue to **AI Agents & Agentic Workflows** in the broader **04. Generative AI** track.